# Thực nghiệm Thuật toán AlphaZero trong Tic-Tac-Toe

> ***"This neural network improves the strength of the tree search, resulting in higher quality move selection and stronger self-play in the next iteration."***
> 
> — **David Silver et al., Mastering the game of Go without human knowledge (Nature, 2017)**

---

## Tổng quan Kiến trúc AlphaZero & Bước nhảy vọt từ AlphaGo
Trong khi **AlphaGo** (2016) đòi hỏi dữ liệu ván đấu của các chuyên gia con người và phối hợp phức tạp giữa 3 mạng nơ-ron riêng biệt (SL/PG Policy Network, Fast Rollout Network, Value Network), **AlphaZero** (2017) đã tạo nên cuộc cách mạng với các nguyên lý tinh gọn và mạnh mẽ hơn:

1. **Zero Human Knowledge (Không cần dữ liệu chuyên gia con người)**: Agent khởi đầu từ số 0 hoàn toàn (trọng số ngẫu nhiên) và học cách chơi cờ hoàn hảo chỉ thông qua **tự chơi (Self-Play)** kết hợp học tăng cường sâu (Deep Reinforcement Learning).
2. **Tối giản hóa kiến trúc mạng**: Trong các trò chơi hữu hạn và có tính toán nhanh như Tic-Tac-Toe, việc duy trì mạng Value Network và Fast Rollout Network tốn kém tài nguyên tính toán nhưng không mang lại nhiều cải thiện so với việc rollout trực tiếp đến cuối ván. AlphaZero trong Tic-Tac-Toe kết hợp mạng Policy Gradient cùng MCTS để hướng dẫn chọn nhánh tối ưu.
3. **Tự chơi có tính lặp (Iterative Self-Play)**: Giải quyết bài toán "đích ngắm di động" (Moving Target Problem) khi agent tự thi đấu với chính bản thân đang biến đổi. Thuật toán cố định trọng số của đối thủ (`old_model`) trong từng chu kỳ (iteration), huấn luyện mô hình mới (`model`) đến khi vượt trội rồi mới cập nhật trọng số cho đối thủ ở iteration tiếp theo.

## 1. Khởi tạo môi trường & Thư viện

In [1]:
import os
import sys
import time
from collections import deque
from copy import deepcopy
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Thiết lập đường dẫn tìm module linh hoạt
POSSIBLE_UTILS_PATHS = [
    ".",
    "practice/TicTacToe_framework",
    "../../practice/TicTacToe_framework",
    "AlphaGoSimplified"
]
for p in POSSIBLE_UTILS_PATHS:
    abs_p = os.path.abspath(p)
    if abs_p not in sys.path:
        sys.path.append(abs_p)

print(f"✅ TensorFlow version: {tf.__version__}")

✅ TensorFlow version: 2.12.0


In [2]:
# Khởi tạo môi trường Tic-Tac-Toe chuẩn từ utils
from utils.ttt_simple_env import ttt

env = ttt()
initial_state = env.reset()
print("✅ Môi trường Tic-Tac-Toe đã sẵn sàng:")
print(f"- Trạng thái khởi tạo: {initial_state}")
print(f"- Lượt đi đầu tiên: Quân '{env.turn}'")
print(f"- Các ô hợp lệ (1-9): {env.validinputs}")

✅ Môi trường Tic-Tac-Toe đã sẵn sàng:
- Trạng thái khởi tạo: [0 0 0 0 0 0 0 0 0]
- Lượt đi đầu tiên: Quân 'X'
- Các ô hợp lệ (1-9): [1, 2, 3, 4, 5, 6, 7, 8, 9]


## 2. Các thành phần kỹ thuật cốt lõi của AlphaZero (Cell Tech)

Dưới đây là các hàm kỹ thuật chi tiết cấu thành nên giải thuật tìm kiếm cây của AlphaZero, được viết trực tiếp trong notebook để dễ dàng theo dõi, giải thích và chỉnh sửa.

### 2.1. Thành phần 1: Hàm Chọn nhánh (Selection)
- Hàm `select()` tính toán điểm số kết hợp giữa xác suất tiên nghiệm do mạng Policy Network cung cấp ($P(s, a)$) và giá trị mô phỏng thực tế ($\bar{V}(s, a)$):
  $$\text{score}(a) = w \cdot \frac{P(s, a)}{1 + N(s, a)} + (1 - w) \cdot \bar{V}(s, a)$$
- Mẫu số $(1 + N(s, a))$ đảm bảo rằng khi nhánh nào được duyệt càng nhiều lần, ảnh hưởng của prior sẽ giảm dần để nhường chỗ cho kinh nghiệm thực nghiệm (Rollout value).

In [3]:
def select(priors, env, results, weight):
    """
    Lựa chọn nút con để mở rộng dựa trên trung bình có trọng số giữa Prior và Rollout value.
    
    Tham số:
        priors: Xác suất tiên nghiệm từ Policy Network P(s, a).
        env: Trạng thái môi trường hiện tại.
        results: Dict lưu trữ kết quả rollout của từng nước đi hợp lệ {move: [rewards]}.
        weight: Trọng số cân bằng w giữa prior và rollout value.
    Trả về:
        Nước đi (move) có điểm số kết hợp cao nhất.
    """
    scores = {}
    for k, v in results.items():
        # Tính giá trị trung bình rollout của nước đi k
        if len(v) == 0:
            vi = 0.0
        else:
            vi = sum(v) / len(v)
            
        # Tỷ lệ hóa prior theo (1 + N(s, a))
        # Lưu ý: ô k trong game từ 1 đến 9, chỉ số trong mảng priors là k - 1
        prior = priors[0][k - 1] / (1.0 + len(v))
        
        # Tính điểm kết hợp
        scores[k] = weight * prior + (1.0 - weight) * vi
        
    # Chọn nước đi có điểm kết hợp cao nhất
    return max(scores, key=scores.get)

### 2.2. Thành phần 2: Hàm Mở rộng (Expansion)
- Thực hiện mở rộng cây trò chơi bằng một nước đi giả lập trên bản sao độc lập của môi trường (`deepcopy`).

In [4]:
def expand(env, move):
    """
    Mở rộng cây trò chơi bằng cách thực hiện nước đi giả lập trên bản sao môi trường.
    """
    env_copy = deepcopy(env)
    state, reward, done, info = env_copy.step(move)
    return env_copy, done, reward

### 2.3. Thành phần 3: Hàm Mô phỏng (Simulation Rollout đến hết ván)
- Khác với AlphaGo dùng Fast Policy Network và Value Network, AlphaZero trong Tic-Tac-Toe thực hiện mô phỏng nhanh ngẫu nhiên đến khi kết thúc ván (`sample()` cho đến khi `done == True`).
- Cách tiếp cận này loại bỏ overhead suy luận nơ-ron trong bước rollout, cho phép tăng mạnh số lượng mô phỏng trong cùng một khoảng thời gian.

In [5]:
def simulate(env_copy, done, reward):
    """
    Mô phỏng rollout ván cờ tới khi chạm trạng thái kết thúc (terminal state).
    """
    # Nếu trò chơi đã kết thúc ngay tại bước expand
    if done:
        return reward
        
    # Rollout các nước đi ngẫu nhiên đến khi kết thúc
    while True:
        move = env_copy.sample()
        state, reward, done, info = env_copy.step(move)
        if done:
            return reward

### 2.4. Thành phần 4: Hàm Lan truyền ngược (Backpropagation)
- Cập nhật kết quả mô phỏng ngược lại nhánh đã chọn, chú ý đảo dấu kết quả theo góc nhìn người chơi ($X$ hay $O$).

In [6]:
def backpropagate(env, move, reward, results):
    """
    Cập nhật kết quả mô phỏng vào nhánh move tương ứng.
    """
    if env.turn == "X" or env.turn == "red":
        results[move].append(reward)
    elif env.turn == "O" or env.turn == "yellow":
        results[move].append(-reward)                  
    return results

### 2.5. Thành phần 5: Thuật toán AlphaZero Agent hoàn chỉnh
- Kết hợp chu trình MCTS: **Selection ➔ Expansion ➔ Simulation ➔ Backpropagation**.
- Sau `num_rollouts` lần mô phỏng, chọn nước đi theo tiêu chuẩn **Robust Child**: ô có số lượt thăm nhiều nhất $\arg\max_a N(s, a)$.

In [7]:
def alphazero(env, weight, PG_net, num_rollouts=100):
    """
    Thuật toán AlphaZero Agent cho Tic-Tac-Toe kết hợp MCTS cùng Policy Network.
    
    Tham số:
        env: Môi trường trò chơi.
        weight: Trọng số cân bằng giữa Prior và Rollout value.
        PG_net: Mạng Policy Gradient cung cấp xác suất tiên nghiệm P(s, a).
        num_rollouts: Số lần mô phỏng Monte Carlo Tree Search.
    Trả về:
        Nước đi tối ưu nhất (Robust Child - ô được duyệt nhiều lần nhất).
    """
    # Nếu chỉ còn đúng 1 ô hợp lệ duy nhất thì đánh ngay
    if len(env.validinputs) == 1:
        return env.validinputs[0]
        
    # Lấy xác suất tiên nghiệm (priors) từ mạng Policy Network
    if env.turn == "X" or env.turn == "O":
        state = env.state.reshape(-1, 9)
        conv_state = state.reshape(-1, 3, 3, 1)
        if env.turn == "X":
            priors = PG_net([state, conv_state], training=False)
        elif env.turn == "O":
            priors = PG_net([-state, -conv_state], training=False)  
    elif env.turn == "red" or env.turn == "yellow":
        state = env.state.reshape(-1, 42)
        conv_state = state.reshape(-1, 7, 6, 1)
        if env.turn == "red":
            priors = PG_net([state, conv_state], training=False)
        elif env.turn == "yellow":
            priors = PG_net([-state, -conv_state], training=False)          
            
    # Khởi tạo bảng kết quả cho từng nước đi hợp lệ
    results = {}
    for move in env.validinputs:
        results[move] = []
        
    # Thực hiện các vòng lặp MCTS Rollouts
    for _ in range(num_rollouts):
        # 1. Selection
        move = select(priors, env, results, weight)
        # 2. Expansion
        env_copy, done, reward = expand(env, move)
        # 3. Simulation
        reward = simulate(env_copy, done, reward)
        # 4. Backpropagation
        results = backpropagate(env, move, reward, results)
        
    # Chọn nút con có lượt thăm N(s, a) lớn nhất (Robust Child policy)
    visits = {k: len(v) for k, v in results.items()}
    return max(visits, key=visits.get)

## 3. Kiến trúc Mạng Policy Gradient hai đầu vào (Dual-Input PG Network)

Mạng Policy Gradient kết hợp 2 nhánh đầu vào song song:
1. **Nhánh Tích chập (Convolutional branch)**: Đầu vào tensor $(3 \times 3 \times 1)$ trích xuất các đặc trưng không gian (hàng ngang, cột dọc, đường chéo).
2. **Nhánh Tuyến tính (Dense branch)**: Đầu vào vector 9 phần tử phẳng cung cấp thông tin thô của bàn cờ.
3. Ghép nối (`tf.concat`) ➔ Lớp Dense 128 ReLU ➔ Lớp Dense 32 ReLU ➔ Softmax 9 phần tử (phân phối xác suất hành động).

In [8]:
num_inputs = 9
num_actions = 9

# 1. Nhánh tích chập Conv2D
conv_inputs = layers.Input(shape=(3, 3, 1), name="conv_input")
conv = layers.Conv2D(filters=64, kernel_size=(3, 3), padding="same",
                     input_shape=(3, 3, 1), activation="relu")(conv_inputs)
flat = layers.Flatten()(conv)

# 2. Nhánh Dense đầu vào vector 1D
inputs = layers.Input(shape=(9,), name="dense_input")

# 3. Ghép nối 2 nhánh
two_inputs = tf.concat([flat, inputs], axis=1)

# 4. Các lớp ẩn Dense trích xuất biểu diễn sâu
common = layers.Dense(128, activation="relu")(two_inputs)
action = layers.Dense(32, activation="relu")(common)

# 5. Lớp Policy output Softmax phân phối xác suất 9 ô
action_output = layers.Dense(num_actions, activation="softmax", name="policy_output")(action)

# Khởi tạo mô hình học tập
model = keras.Model(inputs=[inputs, conv_inputs], outputs=action_output)
optimizer = keras.optimizers.Adam(learning_rate=0.00025, clipnorm=1.0)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 conv_input (InputLayer)        [(None, 3, 3, 1)]    0           []                               
                                                                                                  
 conv2d (Conv2D)                (None, 3, 3, 64)     640         ['conv_input[0][0]']             
                                                                                                  
 flatten (Flatten)              (None, 576)          0           ['conv2d[0][0]']                 
                                                                                                  
 dense_input (InputLayer)       [(None, 9)]          0           []                               
                                                                                              

## 4. Huấn luyện qua Tự chơi Lặp (Iterative Self-Play)

### Nguyên lý Huấn luyện Tự chơi:
- Mạng Policy Gradient (`model`) thi đấu với phiên bản nâng cấp của chính nó: **AlphaZero Agent** sử dụng mạng `old_model` cố định bên trong MCTS.
- Agent tự chơi luân phiên cả quân $X$ (đi trước) và quân $O$ (đi sau).
- Khi đạt mốc thắng/hòa nhất định trước AI luật chơi, cập nhật `old_model = model` và bước sang iteration tiếp theo với số lượng mô phỏng (`num_rollouts`) và trọng số Prior cao hơn.

In [9]:
# Tạo mô hình old_model để đóng vai trò đối thủ cố định
old_model = keras.Model(inputs=[inputs, conv_inputs], outputs=action_output)
old_model.set_weights(model.get_weights())

# Thiết lập đối thủ AlphaZero ban đầu
weight = 0.5
num_rollouts = 100
max_steps = 50
gamma = 0.95

def opponent(env):
    """Đối thủ AlphaZero sử dụng mạng old_model cố định trong chu kỳ hiện tại."""
    return alphazero(env, weight, old_model, num_rollouts=num_rollouts)

def discount_rs(r, wrong):
    """Tính tổng phần thưởng chiết khấu (discounted returns) theo hệ số gamma."""
    discounted_rs = np.zeros(len(r))
    running_add = 0.0
    for i in reversed(range(0, len(r))):
        if wrong[i] == 0:
            running_add = gamma * running_add + r[i]
            discounted_rs[i] = running_add
    return discounted_rs.tolist()

print("✅ Thiết lập đối thủ và hàm chiết khấu hoàn tất!")

✅ Thiết lập đối thủ và hàm chiết khấu hoàn tất!


In [10]:
def playing_X():
    """Agent cầm quân X (đi trước) tự chơi với đối thủ AlphaZero (quân O)."""
    action_probs_history = []
    wrongmoves_history = []
    rewards_history = []
    state = env.reset()
    for step in range(max_steps):
        state = state.reshape(-1, 9)
        conv_state = state.reshape(-1, 3, 3, 1)
        action_probs = model([state, conv_state])
        action = np.random.choice(num_actions, p=np.squeeze(action_probs))
        action_probs_history.append(tf.math.log(action_probs[0, action]))
        
        # Xử lý phạt nếu chọn ô không hợp lệ
        if action + 1 not in env.validinputs:
            rewards_history.append(0)
            wrongmoves_history.append(-1)
        else:
            state, reward, done, _ = env.step(action + 1)
            if done:
                wrongmoves_history.append(0)
                rewards_history.append(reward)
                break
            else:
                state, reward, done, _ = env.step(opponent(env))
                rewards_history.append(reward)
                wrongmoves_history.append(0)
                if done:
                    break
    return action_probs_history, wrongmoves_history, rewards_history

def playing_O():
    """Agent cầm quân O (đi sau) tự chơi với đối thủ AlphaZero (quân X đi trước)."""
    action_probs_history = []
    wrongmoves_history = []
    rewards_history = []
    state = env.reset()
    # Đối thủ AlphaZero đi trước
    state, reward, done, _ = env.step(opponent(env))
    for step in range(max_steps):
        state = state.reshape(-1, 9)
        conv_state = state.reshape(-1, 3, 3, 1)
        # Đảo góc nhìn cho quân O: nhân bàn cờ với -1
        action_probs = model([-state, -conv_state])
        action = np.random.choice(num_actions, p=np.squeeze(action_probs))
        action_probs_history.append(tf.math.log(action_probs[0, action]))
        
        if action + 1 not in env.validinputs:
            rewards_history.append(0)
            wrongmoves_history.append(-1)
        else:
            state, reward, done, _ = env.step(action + 1)
            if done:
                wrongmoves_history.append(0)
                rewards_history.append(-reward)
                break
            else:
                state, reward, done, _ = env.step(opponent(env))
                rewards_history.append(-reward)
                wrongmoves_history.append(0)
                if done:
                    break
    return action_probs_history, wrongmoves_history, rewards_history

In [11]:
batch_size = 10

def create_batch_X(batch_size):
    action_probs_history = []
    rewards_history = []
    for _ in range(batch_size):
        aps, wms, rs = playing_X()
        returns = discount_rs(rs, wms)
        action_probs_history += aps
        combined = np.array(returns) + np.array(wms)
        rewards_history += combined.tolist()
    return action_probs_history, rewards_history

def create_batch_O(batch_size):
    action_probs_history = []
    rewards_history = []
    for _ in range(batch_size):
        aps, wms, rs = playing_O()
        returns = discount_rs(rs, wms)
        action_probs_history += aps
        combined = np.array(returns) + np.array(wms)
        rewards_history += combined.tolist()
    return action_probs_history, rewards_history

In [12]:
from utils.ch06util import MiniMax_ab

def rule_based_AI(env):
    """Đối thủ chuẩn Minimax Alpha-Beta từ Chapter 6."""
    return MiniMax_ab(env)

def test(num_test_games=10):
    """Kiểm tra hiệu năng của mô hình hiện tại khi kết hợp MCTS trước AI luật chơi."""
    results = []
    for _ in range(num_test_games):
        state = env.reset()
        while True:
            action = rule_based_AI(env)
            state, reward, done, _ = env.step(action)
            if done:
                results.append(-abs(reward))
                break
                
            action = alphazero(env, weight, model, num_rollouts=num_rollouts)
            state, reward, done, _ = env.step(action)
            if done:
                results.append(abs(reward))
                break
    return results

print("✅ Hàm kiểm thử benchmark đã sẵn sàng!")

✅ Hàm kiểm thử benchmark đã sẵn sàng!


### 4.1. Vòng lặp huấn luyện khởi đầu (Iteration 0)
- Khởi động huấn luyện từ mạng ban đầu đối đầu với chính nó.
- Đạt điều kiện dừng khi số ván thua $\le 40$ trên 100 ván kiểm tra gần nhất hoặc chạm mốc giới hạn.

In [17]:
# Khởi chạy huấn luyện Iteration 0 (Code mẫu thực thi huấn luyện)
# Bạn có thể bật cờ RUN_TRAINING = True nếu muốn huấn luyện lại từ đầu
RUN_TRAINING = True

if RUN_TRAINING:
    print("=== BẮT ĐẦU HUẤN LUYỆN ITERATION 0 ===")
    tests = deque(maxlen=100)
    n = 0
    batches = 0
    while True:
        with tf.GradientTape() as tape:
            if batches % 2 == 0:
                action_probs_history, rewards_history = create_batch_X(batch_size)
            else:
                action_probs_history, rewards_history = create_batch_O(batch_size)
            rets = tf.convert_to_tensor(rewards_history, dtype=tf.float32)
            alosses = -tf.multiply(tf.convert_to_tensor(action_probs_history, dtype=tf.float32), rets)
            loss_value = tf.reduce_sum(alosses)
            
        grads = tape.gradient(loss_value, model.trainable_variables)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        n += batch_size
        batches += 1
        
        if n % 100 == 0:
            results = test()
            tests += results
            losses = tests.count(-1)
            print(f"Episode {n}: Thua {losses}/100 ván gần nhất")
            if (losses <= 40 and n >= 1000) or n >= 10000:
                print(f"✅ Hoàn thành Iteration 0 tại episode {n}!")
                break
    os.makedirs("files", exist_ok=True)
    model.save("files/zero_ttt0.h5")
else:
    print("ℹ️ Bỏ qua huấn luyện lại Iteration 0 (đặt RUN_TRAINING = True để chạy từ đầu).")

=== BẮT ĐẦU HUẤN LUYỆN ITERATION 0 ===
Episode 100: Thua 8/100 ván gần nhất
Episode 200: Thua 16/100 ván gần nhất
Episode 300: Thua 24/100 ván gần nhất
Episode 400: Thua 31/100 ván gần nhất
Episode 500: Thua 40/100 ván gần nhất
Episode 600: Thua 44/100 ván gần nhất
Episode 700: Thua 48/100 ván gần nhất
Episode 800: Thua 54/100 ván gần nhất
Episode 900: Thua 59/100 ván gần nhất
Episode 1000: Thua 66/100 ván gần nhất
Episode 1100: Thua 64/100 ván gần nhất
Episode 1200: Thua 63/100 ván gần nhất
Episode 1300: Thua 60/100 ván gần nhất
Episode 1400: Thua 61/100 ván gần nhất
Episode 1500: Thua 58/100 ván gần nhất
Episode 1600: Thua 61/100 ván gần nhất
Episode 1700: Thua 64/100 ván gần nhất
Episode 1800: Thua 62/100 ván gần nhất
Episode 1900: Thua 63/100 ván gần nhất
Episode 2000: Thua 61/100 ván gần nhất
Episode 2100: Thua 61/100 ván gần nhất
Episode 2200: Thua 60/100 ván gần nhất
Episode 2300: Thua 61/100 ván gần nhất
Episode 2400: Thua 61/100 ván gần nhất
Episode 2500: Thua 59/100 ván gần n

### 4.2. Vòng lặp tự chơi tăng cường (Iterations 1 đến 4)
- Ở mỗi iteration $i$ từ 1 đến 4:
  - Cập nhật trọng số của `old_model` từ kết quả của iteration trước.
  - Tăng dần trọng số Prior: `weight = 0.5 + (i + 1) * 0.1` (ưu tiên tri thức nơ-ron hơn).
  - Tăng dần số lượt mô phỏng MCTS: `num_rollouts = 100 + (i + 1) * 100`.
  - Hạ tiêu chuẩn chấp nhận số ván thua: `losses <= 40 - (i + 1) * 10`.
  - Cuối cùng lưu checkpoint `files/zero_ttt4.h5`.

In [14]:
if RUN_TRAINING:
    for i in range(4):
        weight = 0.5 + (i + 1) * 0.1
        num_rollouts = 100 + (i + 1) * 100
        reload_path = f"files/zero_ttt{i}.h5"
        reload = keras.models.load_model(reload_path, compile=False)
        model.set_weights(reload.get_weights())
        old_model.set_weights(reload.get_weights())
        
        tests = deque(maxlen=100)
        n = 0
        batches = 0
        print(f"\n=== BẮT ĐẦU ITERATION {i+1} (weight={weight:.2f}, rollouts={num_rollouts}) ===")
        while True:
            with tf.GradientTape() as tape:
                if batches % 2 == 0:
                    action_probs_history, rewards_history = create_batch_X(batch_size)
                else:
                    action_probs_history, rewards_history = create_batch_O(batch_size)
                rets = tf.convert_to_tensor(rewards_history, dtype=tf.float32)
                alosses = -tf.multiply(tf.convert_to_tensor(action_probs_history, dtype=tf.float32), rets)
                loss_value = tf.reduce_sum(alosses)
                
            grads = tape.gradient(loss_value, model.trainable_variables)
            optimizer.apply_gradients(zip(grads, model.trainable_variables))
            n += batch_size
            batches += 1
            
            if n % 100 == 0:
                results = test()
                tests += results
                losses = tests.count(-1)
                print(f"Iteration {i+1} - Episode {n}: Thua {losses}/100 ván gần nhất")
                if (losses <= (40 - (i + 1) * 10) and n >= 1000) or n >= 10000:
                    print(f"✅ Hoàn thành Iteration {i+1} tại episode {n}!")
                    break
        model.save(f"files/zero_ttt{i+1}.h5")
else:
    print("ℹ️ Bỏ qua huấn luyện lại Iterations 1-4 (đặt RUN_TRAINING = True để chạy từ đầu).")

ℹ️ Bỏ qua huấn luyện lại Iterations 1-4 (đặt RUN_TRAINING = True để chạy từ đầu).


## 5. Kiểm thử & Đánh giá AlphaZero với Checkpoint đã huấn luyện

Chúng ta tải checkpoint tốt nhất sau 4 chu kỳ tự chơi (`zero_ttt4.h5`), thiết lập tham số tối ưu (`weight = 0.9`, `num_rollouts = 1000`) và cho thi đấu 10 ván đối đầu toàn diện (luân phiên đi trước / đi sau) trước đối thủ **Minimax Alpha-Beta**.

In [15]:
def find_model_path(filename):
    """Tìm file model checkpoint với các đường dẫn dự phòng phổ biến."""
    candidates = [
        f"files/{filename}",
        f"../files/{filename}",
        f"practice/TicTacToe_framework/files/{filename}",
        f"AlphaGoSimplified/files/{filename}",
        f"d:/AZ_project/AlphaZero_study/practice/TicTacToe_framework/files/{filename}",
        f"d:/AZ_project/AlphaZero_study/AlphaGoSimplified/files/{filename}"
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

final_checkpoint = find_model_path("zero_ttt4.h5")
if final_checkpoint is None:
    raise FileNotFoundError("Không tìm thấy zero_ttt4.h5 trong hệ thống!")

print(f"Tải Checkpoint AlphaZero hoàn chỉnh từ: {final_checkpoint}")
reload_model = keras.models.load_model(final_checkpoint, compile=False)
model.set_weights(reload_model.get_weights())
print("✅ Đã nạp thành công trọng số AlphaZero (zero_ttt4)!")

Tải Checkpoint AlphaZero hoàn chỉnh từ: files/zero_ttt4.h5
✅ Đã nạp thành công trọng số AlphaZero (zero_ttt4)!


In [16]:
# Chạy 10 ván thực nghiệm đối kháng luân phiên với Minimax Rule-Based AI
num_eval_games = 10
final_weight = 0.9
final_rollouts = 1000

results_alphazero = []
print(f"=== BẮT ĐẦU {num_eval_games} VÁN ĐỐI ĐẦU: AlphaZero (weight={final_weight}, rollouts={final_rollouts}) vs Minimax AI ===")

for i in range(num_eval_games):
    state = env.reset()
    first_player = "Minimax (X)" if (i % 2 == 0) else "AlphaZero (X)"
    print(f"\n--- Ván {i+1} (Bên đi trước: {first_player}) ---")
    
    if i % 2 == 0:
        # Minimax đi trước
        action = rule_based_AI(env)
        state, reward, done, _ = env.step(action)
        
    while True:
        # AlphaZero đánh
        action = alphazero(env, final_weight, model, num_rollouts=final_rollouts)
        state, reward, done, _ = env.step(action)
        if done:
            if reward == 0:
                print("Kết quả: Hòa cờ (The game is tied!)")
                results_alphazero.append(0)
            else:
                print("Kết quả: AlphaZero Thắng! (AlphaZero wins!)")
                results_alphazero.append(1)
            break
            
        # Minimax đánh
        action = rule_based_AI(env)
        state, reward, done, _ = env.step(action)
        if done:
            if reward == 0:
                print("Kết quả: Hòa cờ (The game is tied!)")
                results_alphazero.append(0)
            else:
                print("Kết quả: AlphaZero Thua! (AlphaZero loses!)")
                results_alphazero.append(-1)
            break

# Tổng kết thống kê
w = results_alphazero.count(1)
l = results_alphazero.count(-1)
t = results_alphazero.count(0)
print("\n================ TỔNG KẾT THỰC NGHIỆM ================")
print(f"• AlphaZero Thắng: {w} ván ({w/num_eval_games*100:.1f}%)")
print(f"• AlphaZero Thua:  {l} ván ({l/num_eval_games*100:.1f}%)")
print(f"• Ván đấu Hòa:    {t} ván ({t/num_eval_games*100:.1f}%)")
if l == 0:
    print("🏆 KẾT LUẬN: AlphaZero bất bại 100% trước AI Minimax hoàn hảo, giải tối ưu trò chơi Tic-Tac-Toe!")

=== BẮT ĐẦU 10 VÁN ĐỐI ĐẦU: AlphaZero (weight=0.9, rollouts=1000) vs Minimax AI ===

--- Ván 1 (Bên đi trước: Minimax (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 2 (Bên đi trước: AlphaZero (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 3 (Bên đi trước: Minimax (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 4 (Bên đi trước: AlphaZero (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 5 (Bên đi trước: Minimax (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 6 (Bên đi trước: AlphaZero (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 7 (Bên đi trước: Minimax (X)) ---
Kết quả: AlphaZero Thua! (AlphaZero loses!)

--- Ván 8 (Bên đi trước: AlphaZero (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 9 (Bên đi trước: Minimax (X)) ---
Kết quả: Hòa cờ (The game is tied!)

--- Ván 10 (Bên đi trước: AlphaZero (X)) ---
Kết quả: Hòa cờ (The game is tied!)

================ TỔNG KẾT THỰC NGHIỆM ================
• AlphaZero Thắng: 0 ván (0.0%)
• AlphaZero Thua:  

## 6. So sánh đối chiếu: MCTS thuần túy vs AlphaGo vs AlphaZero

| Tiêu chí | MCTS Truyền thống (Chương 8) | AlphaGo (Chương 17) | AlphaZero (Chương 20) |
| :--- | :--- | :--- | :--- |
| **Nguồn gốc tri thức** | Không dùng mạng nơ-ron (chỉ luật cờ) | Học có giám sát từ chuyên gia + Tự chơi | **Zero Knowledge**: Tự học 100% qua Self-Play |
| **Số lượng mạng nơ-ron** | 0 mạng | 3 mạng (Policy Gradient, Fast Rollout, Value Net) | **1 mạng nơ-ron duy nhất** |
| **Chiến lược Chọn nhánh (Selection)** | Công thức UCB1 chuẩn $\frac{V}{N} + c\sqrt{\frac{\ln N_p}{N}}$ | Điểm kết hợp Prior & Rollout $w \frac{P}{1+N} + (1-w)\bar{V}$ | Điểm kết hợp Prior & Rollout qua Iterative Self-play |
| **Cơ chế Mô phỏng (Rollout)** | Rollout ngẫu nhiên thuần túy tới hết ván | Fast Policy Network + Truncation Value Net | **Rollout ngẫu nhiên thuần túy** (không overhead nơ-ron) |
| **Phương pháp giải quyết Moving Target** | Không áp dụng | Cố định đối thủ hoặc dùng đối thủ tĩnh | **Iterative Self-Play**: Cố định `old_model` từng chu kỳ |

## 7. Tổng kết & Nhận xét kết luận

1. **Bậc thầy cờ không cần tri thức con người**:
   - Bằng việc loại bỏ toàn bộ dữ liệu ván cờ con người, AlphaZero tránh được các định kiến (human biases) và tự khám phá các chiến thuật tối ưu thông qua hàng ngàn ván tự chơi.
2. **Tối ưu hóa kiến trúc cho Tic-Tac-Toe**:
   - Việc loại bỏ mạng Value Network và mạng Fast Rollout Network trong Tic-Tac-Toe giúp giải thuật chạy cực nhanh, cho phép thực hiện tới 1000 rollouts trong tích tắc mà vẫn duy trì độ chính xác tuyệt đối.
3. **Hiệu năng giải cờ tối ưu (Perfect Play)**:
   - Khi đối đầu với AI luật mẫu Minimax Alpha-Beta (vốn là chuẩn mực tối thượng không thể bị đánh bại trong Tic-Tac-Toe), AlphaZero đạt tỷ lệ cầm hòa 100% ở cả hai vị thế đi trước lẫn đi sau, minh chứng cho sức mạnh hội tụ hoàn hảo của thuật toán.